# Building Data Genome Project 2 — Fox/Andre (education building) processing

Source: [buds-lab/building-data-genome-project-2](https://github.com/buds-lab/building-data-genome-project-2) (GitHub, Git-LFS files, fetched via `media.githubusercontent.com`). 1,636 buildings across 19 sites; using one building (`Fox_education_Andre`, chosen for having the most complete electricity series among the `Fox` site's 137 buildings) joined with that site's weather.

**This is the weakest-fitting candidate of the 8** and should be flagged as such in the comparison table, for two reasons:

1. The site's raw weather file (`weather.csv`) only has 8 columns (`airTemperature`, `cloudCoverage`, `dewTemperature`, `precipDepth1HR`, `precipDepth6HR`, `seaLvlPressure`, `windDirection`, `windSpeed`) — under the 10-feature minimum on its own, and `precipDepth6HR` is reported on a 6-hour cadence by design (not a data-quality gap), so it was dropped rather than gap-filled at hourly resolution.
2. To clear the 10-feature minimum, 4 cyclical calendar features (`hour_sin/cos`, `dow_sin/cos`) were added. These are legitimate (deterministic functions of the timestamp, known in advance, not lags of the target) but they are engineered, not measured — more than a third of this candidate's feature count is synthetic, unlike every other candidate in this study.

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
import numpy as np
from common import report_candidate, check_cumulative

BASE = "../data/raw/bdg2"
PROCESSED_PATH = "../data/processed/bdg2_fox_andre.csv"
BUILDING = "Fox_education_Andre"

elec = pd.read_csv(f"{BASE}/electricity_cleaned.csv", usecols=["timestamp", BUILDING], parse_dates=["timestamp"])
elec = elec.set_index("timestamp").rename(columns={BUILDING: "electricity_kwh"})

weather = pd.read_csv(f"{BASE}/weather.csv", parse_dates=["timestamp"])
weather = weather[weather.site_id == "Fox"].drop(columns=["site_id", "precipDepth6HR"]).set_index("timestamp")

df = elec.join(weather, how="inner")
df.shape

In [ ]:
# confirm the meter is interval kWh, not a running counter, before treating it as a plain regression target
check_cumulative(df, ["electricity_kwh"])

In [ ]:
TARGET = "electricity_kwh"

df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24)
df["dow_sin"] = np.sin(2 * np.pi * df.index.dayofweek / 7)
df["dow_cos"] = np.cos(2 * np.pi * df.index.dayofweek / 7)

feature_cols = [c for c in df.columns if c != TARGET]
print(len(feature_cols), feature_cols)
(df.isna().mean() * 100).round(2)

`cloudCoverage` (~28% missing) and `windDirection` (~7% missing) carry real reporting gaps, some longer than a few hours. Bounded interpolation + drop, same discipline as every other candidate.

In [ ]:
MAX_GAP_HOURS = 3
before = len(df)
df = df.interpolate(method="time", limit=MAX_GAP_HOURS).dropna(how="any")
print(f"dropped {before - len(df)} rows with unresolved gaps")

In [ ]:
report_candidate(df, TARGET, feature_cols, freq="1h", name="BDG2 Fox/Andre education building (processed)")

In [ ]:
df.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={df.shape}")